# U-Net mining segmentation -> polygons
Continues from MiningDelineationv1.ipynb. Needs:
  model_data/train_patches.npz, model_data/validation_patches.npz, camalan_manifest.csv
Run cell-by-cell in VS Code / Jupyter (the "# %%" markers), or as a plain script.
pip install torch rasterio geopandas shapely scipy pandas scikit-learn

In [1]:
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from scipy.io import loadmat
from PIL import Image


In [2]:
project_dir = Path.cwd()
model_dir = project_dir / "model_data"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)
torch.manual_seed(0); np.random.seed(0)

manifest = pd.read_csv(project_dir / "camalan_manifest.csv")

def load_region(region_id):
    """Same as the notebook: X (H,W,12) float32 in [0,1], y (H,W) uint8."""
    row = manifest.loc[manifest.region_id == region_id].iloc[0]
    a = loadmat(project_dir / row["image_1"])["new"]
    b = loadmat(project_dir / row["image_2"])["new"]
    m = np.asarray(Image.open(project_dir / row["mask"]))
    X = np.concatenate((a, b), axis=-1).astype(np.float32) / 255.0
    return X, (m > 0).astype(np.uint8)

device: cpu


## 1. Data

In [3]:
def load_npz(name):
    d = np.load(model_dir / f"{name}_patches.npz")
    # NHWC -> NCHW for PyTorch
    return np.transpose(d["X"], (0, 3, 1, 2)).copy(), d["y"].astype(np.float32)

Xtr, ytr = load_npz("train")
Xva, yva = load_npz("validation")
print(Xtr.shape, ytr.shape, Xva.shape, yva.shape)
pos_frac = ytr.mean()
print(f"train positive fraction: {pos_frac:.3f}")

class PatchDS(torch.utils.data.Dataset):
    def __init__(self, X, y, augment=False):
        self.X, self.y, self.augment = X, y, augment
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x, y = self.X[i], self.y[i]
        if self.augment:  # same geometric transform on image and mask
            k = np.random.randint(4)
            x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k, (0, 1))
            if np.random.rand() < 0.5:
                x, y = x[:, :, ::-1], y[:, ::-1]
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())[None]

train_dl = torch.utils.data.DataLoader(PatchDS(Xtr, ytr, True), batch_size=16, shuffle=True)
val_dl = torch.utils.data.DataLoader(PatchDS(Xva, yva), batch_size=16)

(355, 12, 128, 128) (355, 128, 128) (36, 12, 128, 128) (36, 128, 128)
train positive fraction: 0.109


## 2. Model

In [6]:
def block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, in_ch=12, base=16, drop=0.2): #12 channels. By using base 16, it is faster than base 32 with minimal loss
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]
        self.enc = nn.ModuleList([block(in_ch, c[0])] + [block(c[i], c[i + 1]) for i in range(4)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(c[i + 1], c[i], 2, stride=2) for i in range(3, -1, -1)])
        self.dec = nn.ModuleList([block(c[i] * 2, c[i]) for i in range(3, -1, -1)])
        self.drop = nn.Dropout2d(drop)
        self.head = nn.Conv2d(c[0], 1, 1)
    def forward(self, x):
        skips = []
        for i, e in enumerate(self.enc):
            x = e(x)
            if i < 4: #4 levels
                skips.append(x); x = F.max_pool2d(x, 2)
        x = self.drop(x)
        for up, dec, s in zip(self.up, self.dec, reversed(skips)):
            x = dec(torch.cat([up(x), s], 1))
        return self.head(x)  # logits

model = UNet().to(device)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M params")

1.94M params


## 3. Train
Mining is only ~4-17% of pixels, so we use BCE (with pos_weight) + Dice.

In [7]:
pos_weight = torch.tensor([(1 - pos_frac) / pos_frac]).sqrt().to(device)  # softened
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def dice_loss(logits, y, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3))
    return (1 - (2 * inter + eps) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + eps)).mean()

def loss_fn(l, y): return bce(l, y) + dice_loss(l, y)

@torch.no_grad()
def evaluate(dl, thr=0.5):
    model.eval()
    tp = fp = fn = 0; vloss = 0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        l = model(x); vloss += loss_fn(l, y).item() * len(x)
        p = (torch.sigmoid(l) > thr).float()
        tp += (p * y).sum().item(); fp += (p * (1 - y)).sum().item(); fn += ((1 - p) * y).sum().item()
    iou = tp / (tp + fp + fn + 1e-9); f1 = 2 * tp / (2 * tp + fp + fn + 1e-9)
    return vloss / len(dl.dataset), iou, f1

EPOCHS = 60
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
best_iou, best_path = 0, model_dir / "unet_best.pt"

for ep in range(1, EPOCHS + 1):
    model.train(); tl = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); l = loss_fn(model(x), y); l.backward(); opt.step()
        tl += l.item() * len(x)
    sched.step()
    vl, iou, f1 = evaluate(val_dl)
    patience, bad = 10, 0 #stops early
    if iou > best_iou:
        bad = 0
        best_iou = iou; torch.save(model.state_dict(), best_path)
    else:
        bad += 1
        if bad >= patience: print("early stop"); break
    print(f"ep {ep:02d} train {tl/len(train_dl.dataset):.3f} | val loss {vl:.3f} IoU {iou:.3f} F1 {f1:.3f}")

model.load_state_dict(torch.load(best_path, map_location=device))

ep 01 train 1.510 | val loss 1.564 IoU 0.006 F1 0.012
ep 02 train 1.350 | val loss 1.448 IoU 0.364 F1 0.533
ep 03 train 1.257 | val loss 1.344 IoU 0.475 F1 0.644
ep 04 train 1.171 | val loss 1.329 IoU 0.350 F1 0.518
ep 05 train 1.090 | val loss 1.184 IoU 0.429 F1 0.601
ep 06 train 1.014 | val loss 1.247 IoU 0.293 F1 0.453
ep 07 train 0.958 | val loss 1.138 IoU 0.486 F1 0.654
ep 08 train 0.882 | val loss 1.031 IoU 0.472 F1 0.641
ep 09 train 0.825 | val loss 1.052 IoU 0.503 F1 0.670
ep 10 train 0.777 | val loss 0.937 IoU 0.545 F1 0.705
ep 11 train 0.734 | val loss 0.924 IoU 0.504 F1 0.670
ep 12 train 0.696 | val loss 0.917 IoU 0.529 F1 0.692
ep 13 train 0.658 | val loss 0.924 IoU 0.440 F1 0.612
ep 14 train 0.618 | val loss 0.819 IoU 0.528 F1 0.691
ep 15 train 0.599 | val loss 0.803 IoU 0.506 F1 0.672
ep 16 train 0.575 | val loss 0.783 IoU 0.559 F1 0.717
ep 17 train 0.538 | val loss 0.793 IoU 0.539 F1 0.700
ep 18 train 0.517 | val loss 0.732 IoU 0.533 F1 0.695
ep 19 train 0.531 | val loss

<All keys matched successfully>

## 4. Pick a threshold on validation (never on test)

In [8]:
thrs = np.arange(0.2, 0.81, 0.05)
scores = [evaluate(val_dl, t)[1] for t in thrs]
THR = float(thrs[int(np.argmax(scores))])
print("best threshold:", THR, "val IoU:", max(scores))

best threshold: 0.7 val IoU: 0.6137370158627662


## 5. Full-region prediction (sliding window, overlap-averaged)

In [10]:
@torch.no_grad()
def predict_region(region_id, patch=128, stride=64, batch=16):
    X, y = load_region(region_id)
    H, W, _ = X.shape
    Xc = np.transpose(X, (2, 0, 1))
    pad_h, pad_w = max(0, patch - H), max(0, patch - W)
    Xc = np.pad(Xc, ((0, 0), (0, pad_h), (0, pad_w)), mode="reflect")
    _, Hp, Wp = Xc.shape
    rows = list(range(0, Hp - patch + 1, stride)); cols = list(range(0, Wp - patch + 1, stride))
    if rows[-1] != Hp - patch: rows.append(Hp - patch)
    if cols[-1] != Wp - patch: cols.append(Wp - patch)
    coords = [(r, c) for r in rows for c in cols]
    acc = np.zeros((Hp, Wp), np.float32); cnt = np.zeros((Hp, Wp), np.float32)
    model.eval()
    for i in range(0, len(coords), batch):
        cb = coords[i:i + batch]
        xb = torch.from_numpy(np.stack([Xc[:, r:r + patch, c:c + patch] for r, c in cb])).to(device)
        pb = torch.sigmoid(model(xb)).cpu().numpy()[:, 0]
        for (r, c), p in zip(cb, pb):
            acc[r:r + patch, c:c + patch] += p; cnt[r:r + patch, c:c + patch] += 1
    return (acc / cnt)[:H, :W], y

def metrics(prob, y, thr):
    p = prob > thr
    tp = (p & (y == 1)).sum(); fp = (p & (y == 0)).sum(); fn = (~p & (y == 1)).sum()
    return dict(IoU=tp / (tp + fp + fn + 1e-9), F1=2 * tp / (2 * tp + fp + fn + 1e-9),
                precision=tp / (tp + fp + 1e-9), recall=tp / (tp + fn + 1e-9))

# Held-out test regions (12-15): full-image evaluation
test_ids = manifest.loc[manifest.split == "test", "region_id"].astype(int).tolist()
rows = []
for rid in test_ids:
    prob, y = predict_region(rid)
    rows.append({"region_id": rid, **metrics(prob, y, THR)})
print(pd.DataFrame(rows).round(3))

   region_id    IoU     F1  precision  recall
0         12  0.617  0.763      0.721   0.809
1         13  0.562  0.719      0.755   0.687
2         14  0.767  0.868      0.840   0.899
3         15  0.772  0.871      0.831   0.916


## 6. Mask -> polygons (GeoJSON)
Cleans the binary mask, vectorizes with rasterio, filters tiny polygons in a metric CRS
(UTM 19S, EPSG:32719, covers Madre de Dios), and writes EPSG:4326.

In [13]:
import rasterio.features
from rasterio.transform import from_bounds
import geopandas as gpd
from shapely.geometry import shape
from scipy import ndimage as ndi

def mask_to_polygons(prob, region_id, thr=THR, min_area_m2=5000, simplify_m=10):
    m = prob > thr
    m = ndi.binary_opening(m, iterations=1)
    m = ndi.binary_closing(m, iterations=2)
    m = ndi.binary_fill_holes(m)
    row = manifest.loc[manifest.region_id == region_id].iloc[0]
    tf = from_bounds(row.west, row.south, row.east, row.north, m.shape[1], m.shape[0])
    geoms = [shape(g) for g, v in rasterio.features.shapes(m.astype(np.uint8), mask=m, transform=tf) if v == 1]
    gdf = gpd.GeoDataFrame({"region_id": [region_id] * len(geoms)}, geometry=geoms, crs="EPSG:4326") #thanks claude for helping me debug the code it wrote originally
    if gdf.empty: return gdf
    gdf = gdf.to_crs("EPSG:32719")
    gdf = gdf[gdf.area >= min_area_m2].copy()
    gdf["geometry"] = gdf.geometry.simplify(simplify_m)
    gdf["area_ha"] = gdf.area / 1e4
    gdf["mean_prob"] = np.nan  # fill below if wanted
    return gdf.to_crs("EPSG:4326")

out_dir = project_dir / "predicted_polygons"; out_dir.mkdir(exist_ok=True) #I have changed the following code from geojson to gpkg for easy use in arcgis
all_polys = []
for rid in test_ids + [16]:
    prob, _ = predict_region(rid)
    g = mask_to_polygons(prob, rid)
    print(f"region {rid}: {len(g)} polygons")
    if len(g):
        all_polys.append(g)

if all_polys:
    allg = pd.concat(all_polys, ignore_index=True)
    allg.to_file(out_dir / "predicted_mining.gpkg", layer="mining", driver="GPKG")  # for ArcGIS Pro

region 12: 150 polygons
region 13: 178 polygons
region 14: 285 polygons
region 15: 650 polygons
region 16: 78 polygons
